[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_13/NB_bishop_ch13_gnn_demo.ipynb)

# Chapter 13: Graph Neural Networks -- Message Passing Demo

**Bishop & Bishop (2024), Chapter 13**

This notebook implements GNN message passing from scratch in pure NumPy,
demonstrating how node embeddings evolve as information propagates through the graph.

**Course:** Neural Networks and Deep Learning with Business Applications

**Author:** Daniel Traian PELE -- Bucharest University of Economic Studies

In [ ]:
# Run in Google Colab to install dependencies
!pip install -q networkx


In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx

# Course colors
BLUE    = '#1f4e79'
CRIMSON = '#c00000'
GREEN   = '#2e7d32'

# Matplotlib styling
matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor']   = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['figure.figsize'] = (10, 5)

np.random.seed(42)

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Chapter 13: GNN Message Passing Demo -- Ready!')

## 1. Create a Simple Graph

We use Zachary's Karate Club -- a classic 34-node social network with two
ground-truth communities. Each node starts with a one-hot feature vector.

In [ ]:
# Load Karate Club
G = nx.karate_club_graph()
N = G.number_of_nodes()
A = nx.to_numpy_array(G)

# Ground-truth labels (Mr. Hi = 0, Officer = 1)
labels = np.array([G.nodes[i]['club'] == 'Officer' for i in range(N)], dtype=int)

# Initial node features: one-hot identity
H0 = np.eye(N, dtype=np.float64)

print(f'Karate Club: {N} nodes, {G.number_of_edges()} edges')
print(f'Adjacency matrix shape: {A.shape}')
print(f'Initial features H0 shape: {H0.shape}')
print(f'Community 0 (Mr. Hi):  {(labels == 0).sum()} nodes')
print(f'Community 1 (Officer): {(labels == 1).sum()} nodes')

In [ ]:
# Visualize the graph
pos = nx.spring_layout(G, seed=42)
node_colors = [BLUE if labels[i] == 0 else CRIMSON for i in range(N)]

fig, ax = plt.subplots(figsize=(9, 7))
nx.draw(G, pos, ax=ax, node_color=node_colors, node_size=350,
        with_labels=True, font_size=8, font_color='white',
        edge_color='gray', width=0.6)
ax.set_title("Zachary's Karate Club -- Two Ground-Truth Communities", fontsize=13)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=BLUE,    markersize=10, label='Mr. Hi (0)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=CRIMSON, markersize=10, label='Officer (1)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
fig.tight_layout()
save_fig(fig, 'bishop_ch13_gnn_demo_graph')
plt.show()

## 2. Message Passing in Pure NumPy

A single GCN-style message-passing round performs:

$$H^{(l+1)} = \sigma\!\left(\tilde{D}^{-1/2}\,\tilde{A}\,\tilde{D}^{-1/2}\; H^{(l)}\, W^{(l)}\right)$$

where $\tilde{A} = A + I$ (adjacency with self-loops) and $\tilde{D}$ is its
degree matrix. We implement this step by step.

In [ ]:
def gcn_normalize(A):
    """Symmetric normalization: D^{-1/2} A_hat D^{-1/2} with self-loops."""
    N = A.shape[0]
    A_hat = A + np.eye(N)
    deg = A_hat.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg))
    return D_inv_sqrt @ A_hat @ D_inv_sqrt


def message_pass(A_norm, H, W, use_relu=True):
    """One round of GCN message passing: aggregate neighbours, transform, activate."""
    # Step 1 -- aggregate: each node collects weighted features from neighbours
    H_agg = A_norm @ H
    # Step 2 -- linear transform
    H_out = H_agg @ W
    # Step 3 -- non-linearity
    if use_relu:
        H_out = np.maximum(0, H_out)
    return H_out


A_norm = gcn_normalize(A)
print('Normalized adjacency computed.')
print(f'Row sums (should be close to 1): {A_norm.sum(axis=1)[:5].round(4)}')

In [ ]:
# Demonstrate aggregation for a single node
node_id = 0
neighbors = list(G.neighbors(node_id))
weights = A_norm[node_id]
nonzero = np.where(weights > 1e-8)[0]

print(f'Node {node_id} has {len(neighbors)} neighbours: {neighbors}')
print(f'\nAggregation weights (including self-loop):')
for j in nonzero:
    tag = '(self)' if j == node_id else ''
    print(f'  w[{node_id},{j:2d}] = {weights[j]:.4f}  {tag}')
print(f'\nSum of weights: {weights[nonzero].sum():.4f}')

## 3. Embedding Evolution over 1, 2, 3 Rounds

We apply successive message-passing rounds and project the resulting
node embeddings to 2-D (via PCA) to see how they cluster.

In [ ]:
from sklearn.decomposition import PCA

np.random.seed(42)

# Random weight matrices for each layer  (34 -> 16 -> 8 -> 4)
dims = [N, 16, 8, 4]
Ws = [np.random.randn(dims[i], dims[i+1]) * 0.3 for i in range(3)]

# Collect embeddings after 0, 1, 2, 3 rounds
embeddings = {0: H0.copy()}
H = H0.copy()
for r in range(1, 4):
    H = message_pass(A_norm, H, Ws[r-1], use_relu=True)
    embeddings[r] = H.copy()
    print(f'Round {r}: embedding shape = {H.shape}')

# PCA to 2-D for each round
coords = {}
for r, emb in embeddings.items():
    pca = PCA(n_components=2, random_state=42)
    coords[r] = pca.fit_transform(emb)

print('\nPCA projections computed for all rounds.')

In [ ]:
# Plot embedding evolution
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for r, ax in enumerate(axes):
    c2 = coords[r]
    for lab, color, name in [(0, BLUE, 'Mr. Hi'), (1, CRIMSON, 'Officer')]:
        mask = labels == lab
        ax.scatter(c2[mask, 0], c2[mask, 1], c=color, s=70,
                   edgecolors='white', linewidths=0.5, label=name, alpha=0.85)
    ax.set_title(f'Round {r}' if r > 0 else 'Round 0 (initial)', fontsize=11)
    ax.set_xlabel('PC 1')
    if r == 0:
        ax.set_ylabel('PC 2')
    ax.legend(fontsize=8)

fig.suptitle('Node Embeddings After Successive Message-Passing Rounds (PCA)',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'bishop_ch13_gnn_demo_embedding_evolution')
plt.show()

## 4. Graph Visualization with Embedding-Based Node Colors

We color each node by its first principal-component value after 3 rounds
of message passing. Nodes in the same community should receive similar colors.

In [ ]:
# Color by first PC of round-3 embedding
pc1 = coords[3][:, 0]
pc1_norm = (pc1 - pc1.min()) / (pc1.max() - pc1.min())  # scale to [0, 1]

# Build a custom blue-red colormap
from matplotlib.colors import LinearSegmentedColormap
cmap_br = LinearSegmentedColormap.from_list('course', [BLUE, 'white', CRIMSON])

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# Left: ground-truth labels
gt_colors = [BLUE if labels[i] == 0 else CRIMSON for i in range(N)]
nx.draw(G, pos, ax=axes[0], node_color=gt_colors, node_size=350,
        with_labels=True, font_size=8, font_color='white',
        edge_color='gray', width=0.5)
axes[0].set_title('Ground-Truth Communities', fontsize=12)

# Right: embedding-based colors
nx.draw(G, pos, ax=axes[1], node_color=pc1_norm, cmap=cmap_br,
        node_size=350, with_labels=True, font_size=8, font_color='black',
        edge_color='gray', width=0.5, vmin=0, vmax=1)
axes[1].set_title('Node Color = PC1 After 3 Message-Passing Rounds', fontsize=12)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_br, norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[1], shrink=0.6, label='PC1 (normalised)')

fig.tight_layout()
save_fig(fig, 'bishop_ch13_gnn_demo_graph_colors')
plt.show()

In [ ]:
# Quantitative: cosine similarity within vs between communities after 3 rounds
from numpy.linalg import norm

emb3 = embeddings[3]
# Normalise rows
emb3_n = emb3 / (norm(emb3, axis=1, keepdims=True) + 1e-12)
cos_sim = emb3_n @ emb3_n.T

intra_0 = cos_sim[np.ix_(labels == 0, labels == 0)]
intra_1 = cos_sim[np.ix_(labels == 1, labels == 1)]
inter   = cos_sim[np.ix_(labels == 0, labels == 1)]

# Exclude self-similarity from intra
intra_vals = np.concatenate([
    intra_0[np.triu_indices_from(intra_0, k=1)],
    intra_1[np.triu_indices_from(intra_1, k=1)]
])
inter_vals = inter.flatten()

print('Cosine similarity after 3 rounds of message passing')
print(f'  Intra-community (same group):     mean = {intra_vals.mean():.4f}')
print(f'  Inter-community (different group): mean = {inter_vals.mean():.4f}')
print(f'  Separation gap:                    {intra_vals.mean() - inter_vals.mean():.4f}')

## Key Takeaways

1. **Message passing** is the core operation of GNNs: each node aggregates
   features from its neighbours and then applies a learned linear transformation.

2. **Self-loops** ($\tilde{A} = A + I$) ensure that a node retains its own
   features during aggregation.

3. **Symmetric normalisation** ($\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$)
   prevents high-degree nodes from dominating the aggregation.

4. After just **2--3 rounds**, node embeddings already capture community
   structure: intra-community cosine similarity is significantly higher
   than inter-community similarity.

5. The **receptive field** of a node grows with each round -- after $k$ rounds
   a node has information from all nodes within $k$ hops.

In [ ]:
takeaways = [
    '1. Message passing = aggregate neighbour features + linear transform + activation.',
    '2. Self-loops ensure a node keeps its own information during aggregation.',
    '3. Symmetric normalisation balances contributions from high- and low-degree nodes.',
    '4. After 2-3 rounds, embeddings cluster by community (high intra, low inter similarity).',
    '5. Receptive field grows with each round: k rounds = k-hop neighbourhood.'
]
print('Key Takeaways')
print('=' * 60)
for t in takeaways:
    print(t)